LIBRARIES

In [30]:
#pip install torchvision

In [31]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.optim as optim

INPUTS

In [32]:
IMG_HEIGHT= 227
IMG_WIDTH= 227
IMG_CHANNELS=3
CLASS_NAMES = ["lilly", "lotus", "orchid", "sunflower", "tulip"]
learning_rate = 0.0001

In [33]:
transform= transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dir = r"C:\My Folder\Projects\Computer_Vision\flower_images\train"
val_dir = r"C:\My Folder\Projects\Computer_Vision\flower_images\val"

train_data = datasets.ImageFolder(root=train_dir, transform= transform)
val_data = datasets.ImageFolder(root=val_dir, transform= transform)

train_dataset= DataLoader(train_data, batch_size= 16, shuffle=True)
val_dataset = DataLoader(val_data, batch_size= 16, shuffle=True)

NETWORK ARCHITECTURE

In [34]:
alexnet = models.alexnet(pretrained= True)

for params in alexnet.parameters():
    params.requires_grad =False

alexnet.classifier[6]= nn.Linear(4096, 5)

for params in alexnet.classifier[6].parameters():
    params.requires_grad=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alexnet = alexnet.to(device)


C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=learning_rate)

In [36]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs ):
    for epoch in range(epochs):
        correct_preds=0
        total_samples=0
        total_loss = 0
    
        model.train()
        for images, labels in train_loader:
            images, labels =  images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs=model(images)
            loss=criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss+=loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct= (preds==labels).sum().item()
            correct_preds += batch_correct
            total_samples +=labels.size(0)

        train_acc= correct_preds/total_samples    
        print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {train_acc:.4f}")

        model.eval()
        val_correct = 0
        val_total =0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels =  images.to(device), labels.to(device)
                outputs = model(images)
            
                _, preds = torch.max(outputs, 1)
            
                val_correct +=(preds==labels).sum().item()
                val_total+=labels.size(0)

            val_acc= val_correct/val_total
            print(f"Val_Accuracy:{val_acc:.4f}")



TRAINING

In [37]:
epochs =10
train_model(alexnet, criterion, optimizer, train_dataset, val_dataset, epochs)

Epoch: 1, Loss: 184.5737, Accuracy: 0.6237
Val_Accuracy:0.7960
Epoch: 2, Loss: 108.1579, Accuracy: 0.8013
Val_Accuracy:0.8320
Epoch: 3, Loss: 92.1052, Accuracy: 0.8250
Val_Accuracy:0.8580
Epoch: 4, Loss: 79.8717, Accuracy: 0.8573
Val_Accuracy:0.8570
Epoch: 5, Loss: 73.0161, Accuracy: 0.8680
Val_Accuracy:0.8780
Epoch: 6, Loss: 67.0953, Accuracy: 0.8780
Val_Accuracy:0.8810
Epoch: 7, Loss: 62.0227, Accuracy: 0.8927
Val_Accuracy:0.8870
Epoch: 8, Loss: 58.9049, Accuracy: 0.8967
Val_Accuracy:0.8890
Epoch: 9, Loss: 55.8900, Accuracy: 0.9033
Val_Accuracy:0.8900
Epoch: 10, Loss: 52.9494, Accuracy: 0.9110
Val_Accuracy:0.8970
